In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')
print(os.getcwd())

/content/drive/MyDrive/credit-risk-assessment-system


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

Step 2 — load the data and split immediately, before touching anything else

In [7]:
df = pd.read_csv('data/raw/application_train.csv')

X = df.drop(columns=['TARGET'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

Train shape: (246008, 121)
Test shape: (61503, 121)
Train default rate: 0.08072908198107379
Test default rate: 0.08072776937710356


Step 3 — handle missing values

3a — Drop columns that are missing too much to be useful

In [8]:
missing_pct = X_train.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct > 50].index.tolist()

print(f"Dropping {len(high_missing_cols)} columns with >50% missing:")
print(high_missing_cols)

Dropping 41 columns with >50% missing:
['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']


In [9]:
X_train = X_train.drop(columns=high_missing_cols)
X_test = X_test.drop(columns=high_missing_cols)

print(X_train.shape, X_test.shape)

(246008, 80) (61503, 80)


3b — For remaining missing values, split by column type

In [10]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

print(f"{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns")

67 numeric columns, 13 categorical columns


In [11]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

In [12]:
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

Step 4 — verify no missing values remain

In [13]:
print("Train missing values:", X_train.isnull().sum().sum())
print("Test missing values:", X_test.isnull().sum().sum())

Train missing values: 0
Test missing values: 0


Step 5 — encode categorical columns

In [14]:
print(f"Before encoding: {X_train.shape[1]} columns")

X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

print(f"After encoding: {X_train_encoded.shape[1]} columns")

Before encoding: 80 columns
After encoding: 180 columns


In [15]:
train_cols = set(X_train_encoded.columns)
test_cols = set(X_test_encoded.columns)

print("In train but not test:", train_cols - test_cols)
print("In test but not train:", test_cols - train_cols)

In train but not test: {'NAME_FAMILY_STATUS_Unknown'}
In test but not train: set()


In [16]:
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print(X_train_encoded.shape, X_test_encoded.shape)

(246008, 180) (61503, 180)


In [17]:
train_cols = set(X_train_encoded.columns)
test_cols = set(X_test_encoded.columns)

print("In train but not test:", train_cols - test_cols)
print("In test but not train:", test_cols - train_cols)

In train but not test: set()
In test but not train: set()


Step 6 — feature scaling

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_encoded[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test_encoded[numeric_cols])

Step 7 — save the processed data so the next notebook can use it directly

In [19]:
X_train_scaled.to_csv('data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print("Saved processed data:")
print(X_train_scaled.shape, X_test_scaled.shape)

Saved processed data:
(246008, 180) (61503, 180)
